##### Loading the Data

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/raw/construction_dataset.csv')

print("--- Dataset Columns ---")
print(df.columns.tolist())
display(df.head())

print("\n--- Missing Values ---")
print(df.isnull().sum())

--- Dataset Columns ---
['Task_ID', 'Task_Duration_Days', 'Labor_Required', 'Equipment_Units', 'Material_Cost_USD', 'Start_Constraint', 'Risk_Level', 'Resource_Constraint_Score', 'Site_Constraint_Score', 'Dependency_Count']


,Task_ID,Task_Duration_Days,Labor_Required,Equipment_Units,Material_Cost_USD,Start_Constraint,Risk_Level,Resource_Constraint_Score,Site_Constraint_Score,Dependency_Count
0,T1,52,14,6,16789.73,0,Medium,0.41,0.59,4
1,T2,15,2,2,16885.80,5,Low,0.75,0.17,3
2,T3,72,11,1,7978.70,22,Low,0.96,0.41,1
3,T4,61,1,5,19379.02,18,Low,0.41,0.67,4
4,T5,21,19,5,66757.72,22,Low,0.85,0.63,3



--- Missing Values ---
Task_ID                      0
Task_Duration_Days           0
Labor_Required               0
Equipment_Units              0
Material_Cost_USD            0
Start_Constraint             0
Risk_Level                   0
Resource_Constraint_Score    0
Site_Constraint_Score        0
Dependency_Count             0
dtype: int64


##### Creating Actual Values and EVM Metrics

In [3]:
# Simulate realistic 'Actual' values using the real dataset
np.random.seed(42)

# Actual Duration = Baseline + realistic slippage based on risk
risk_multiplier = {'Low': 1.1, 'Medium': 1.3, 'High': 1.6}
df['Actual_Duration'] = df.apply(
    lambda row: row['Task_Duration_Days'] * risk_multiplier[row['Risk_Level']] 
    * np.random.uniform(0.85, 1.15), axis=1
)

# Actual Cost = Baseline Cost + realistic overrun
df['Actual_Cost'] = df['Material_Cost_USD'] * np.random.uniform(0.9, 1.3, size=len(df))

# EVM Metrics
df['SPI'] = df['Task_Duration_Days'] / df['Actual_Duration']
df['CPI'] = df['Material_Cost_USD'] / df['Actual_Cost']

# Flag performance
df['Schedule_Status'] = df['SPI'].apply(
    lambda x: 'Critical' if x < 0.8 else ('Stakeholder Alert' if x < 1.0 else 'Healthy')
)

##### Optimism Bias Detection

In [ ]:
# Calculate the gap between planned and actual
df['Duration_Gap'] = df['Actual_Duration'] - df['Task_Duration_Days']

# Overall optimism bias
avg_planned = df['Task_Duration_Days'].mean()
avg_actual = df['Actual_Duration'].mean()
bias = avg_actual - avg_planned

print(f"Average Planned Duration: {avg_planned:.1f} days")
print(f"Average Actual Duration:  {avg_actual:.1f} days")
print(f"Optimism Bias:            {bias:.1f} days")

# Bias by risk level
bias_by_risk = df.groupby('Risk_Level').agg(
    Avg_Planned=('Task_Duration_Days', 'mean'),
    Avg_Actual=('Actual_Duration', 'mean')
).round(1)

bias_by_risk['Bias_Days'] = (
    bias_by_risk['Avg_Actual'] - bias_by_risk['Avg_Planned']
).round(1)

print("\n--- Optimism Bias by Risk Level ---")
print(bias_by_risk.to_string())

Average Planned Duration: 43.4 days
Average Actual Duration:  54.5 days
Optimism Bias:            11.1 days

--- Optimism Bias by Risk Level ---
            Avg_Planned  Avg_Actual  Bias_Days
Risk_Level                                    
High               43.9        69.6       25.7
Low                43.5        47.5        4.0
Medium             43.1        56.1       13.0


##### The Random Forest Forecasting Model

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# --- 1. PREPARE FEATURES ---
X = df.iloc[:, [3, 4]]
y = df['Actual_Duration']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- 2. TRAIN THE FORECASTER ---
forecaster = RandomForestRegressor(n_estimators=100, random_state=42)
forecaster.fit(X_train, y_train)

# --- 3. VALIDATE ---
predictions = forecaster.predict(X_test)
mae = mean_absolute_error(y_test, predictions)

print(f"Forecasting Accuracy: The model can predict completion dates within +/- {mae:.2f} days.")

# --- 4. THE DECISION SUPPORT TOOL ---
new_task = pd.DataFrame([[30, 50000]], columns=X.columns)
predicted_delay = forecaster.predict(new_task)[0]

print(f"\n💡 PROJECT PLANNER INSIGHT:")
print(f"For a new 30-day task, the model predicts a real-world duration of {predicted_delay:.1f} days.")
print(f"Recommendation: Adjust baseline by {predicted_delay - 30:.1f} days to ensure schedule credibility.")

Forecasting Accuracy: The model can predict completion dates within +/- 31.73 days.

💡 PROJECT PLANNER INSIGHT:
For a new 30-day task, the model predicts a real-world duration of 92.1 days.
Recommendation: Adjust baseline by 62.1 days to ensure schedule credibility.
